In [1]:
!pip install -q "ultralytics==8.4.149"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 460.4 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from zipfile import ZipFile
import torch

from ultralytics import SAM
import ultralytics

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

Ultralytics: 8.4.149
PyTorch: 2.11.0+cu128
Device: 0


**Paths**

In [5]:
DATASET_URL = (
    "https://github.com/ultralytics/"
    "assets/releases/download/v0.0.0/"
    "crack-seg.zip"
)

DATASETS_ROOT = Path("/content/datasets")
ARCHIVE_PATH = DATASETS_ROOT / "crack-seg.zip"
DATASET_ROOT = DATASETS_ROOT / "crack-seg"

In [6]:
DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {
    "train": 3717,
    "val": 200,
    "test": 112,
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

print("Dataset root:", DATASET_ROOT)

Dataset root: /content/datasets/crack-seg


In [7]:
dataset_ready = (DATASET_ROOT / "images" / "train").is_dir()

if not dataset_ready:
    if not ARCHIVE_PATH.is_file():
        print("Downloading Crack-Seg...")
        torch.hub.download_url_to_file(
            DATASET_URL, str(ARCHIVE_PATH), progress=True
        )
        
    print("Extracting dataset...")
    with ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(DATASETS_ROOT)


if not DATASET_ROOT.is_dir():
    candidates = [
        path for path in DATASETS_ROOT.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and "crack" in path.name.lower()
    ]

    if len(candidates) != 1:
        candidates = [
            path for path in DATASETS_ROOT.rglob("images")
            if path.is_dir()
        ]
        if len(candidates) == 1:
            candidates = [candidates[0].parent]

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not locate Crack-Seg root in {DATASETS_ROOT}. "
            f"Found candidates: {candidates}"
        )

    DATASET_ROOT = candidates[0]

100%|██████████| 91.6M/91.6M [00:03<00:00, 24.4MB/s]


Extracting dataset...


In [8]:
for split in SPLITS:
    required_directories = [
        (DATASET_ROOT / "images" / split),
        (DATASET_ROOT / "labels" / split)
    ]
    
    for directory in (required_directories):
        if not directory.is_dir():
            raise FileExistsError(directory)

print("Dataset extracted:", DATASET_ROOT)

Dataset extracted: /content/datasets


In [9]:
VAL_IMAGES_DIR = DATASET_ROOT/ "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_05/"
    "sam2_image"
)

FIGURE_DIR =  OUTPUT_ROOT / "figures"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


print("Validation images:", VAL_IMAGES_DIR)
print("Validation labels:", VAL_LABELS_DIR)
print("Outputs:", OUTPUT_ROOT)

Validation images: /content/datasets/images/val
Validation labels: /content/datasets/labels/val
Outputs: /content/drive/MyDrive/vision_unit_02_outputs/block_05/sam2_image


**Image index**

In [10]:
validation_image_index = {
    path.stem: path 
    for path in VAL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
}

print("Validation images:", len(validation_image_index))

Validation images: 200


**Polygon → separate instance masks**

In [11]:
def load_instance_masks(image_path, label_path):
    bgr_image = cv2.imread(str(image_path))
    
    if bgr_image is None:
        raise ValueError(f"Cannot read: {image_path}")

    rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    height, width = rgb_image.shape[:2]
    
    instance_masks = []
    
    label_text = label_path.read_text(encoding="utf-8").strip()
    if not label_text:
        return rgb_image, instance_masks
    
    for line_number, line in enumerate(label_text.splitlines(), start=1):
        values = line.split()
        
        if (len(values) < 7 or (len(values) - 1) % 2 != 0):
            raise ValueError(
                f"Invalid polygon: "
                f"{label_path.name}, "
                f"line {line_number}"
            )
        
        class_id = int(float(values[0]))
        if class_id != 0:
             raise ValueError(
                f"Unexpected class {class_id}"
            )
        
        normalized_points = np.asarray(values[1:], dtype=np.float32).reshape(-1, 2)
        
        if not np.all((normalized_points >= 0)& (normalized_points <= 1)):
            raise ValueError(
                f"Out-of-bounds polygon: "
                f"{label_path.name}"
            )
        
        pixel_points = np.empty_like(
            normalized_points, dtype=np.int32
        )
        
        pixel_points[:, 0] = np.clip(
            np.rint(normalized_points[:, 0] * width), 0, 
            width - 1
        ).astype(np.int32)
        
        pixel_points[:, 1] = np.clip(
            np.rint(normalized_points[:, 1] * height), 0, 
            height - 1
        ).astype(np.int32)
        
        mask = np.zeros((height, width), dtype=np.uint8)
        
        cv2.fillPoly(mask, [pixel_points], color=1)
        if mask.any():
            instance_masks.append(mask)
    
    return rgb_image, instance_masks

**Prompt and metric utilities**

In [12]:
def binary_iou(prediction, target):
    prediction = prediction > 0
    target = target > 0
    
    intersection = np.logical_and(prediction, target).sum()
    union = np.logical_or(prediction, target).sum()
    
    if union == 0:
        return np.nan
    
    return float(intersection / union)

In [13]:
def binary_dice(prediction, target):
    prediction = prediction > 0
    target = target > 0
    
    intersection = np.logical_and(prediction, target).sum()
    denominator = (prediction.sum + target.sum())
    
    if denominator == 0:
        return np.nan

    return float(2 * intersection / denominator)

**Positive-point selection**

In [14]:
def find_positive_point(mask):
    distance = cv2.distanceTransform(
        mask.astype(np.uint8), cv2.DIST_L2, 5
    )
    
    y, x = np.unravel_index(
        np.argmax(distance), distance.shape
    )
    
    return int(x), int(y)

In [15]:
def mask_to_box(mask):
    ys, xs = np.where(mask > 0)
    
    if len(xs) == 0:
        raise ValueError("Empty instance mask.")
    
    return [
        int(xs.min()),
        int(ys.min()),
        int(xs.max()),
        int(ys.max()),
    ]

In [16]:
def find_negative_point(mask, padding=15,):
    height, width = mask.shape
    x1, y1, x2, y2 = mask_to_box(mask)

    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(width - 1, x2 + padding)
    y2 = min(height - 1, y2 + padding)

    background = (mask == 0).astype(np.uint8)

    distance = cv2.distanceTransform(
        background,cv2.DIST_L2, 5
    )

    valid_region = np.zeros_like(
        mask, dtype=bool
    )

    valid_region[y1:y2 + 1, x1:x2 + 1] = True
    valid_region &= (mask == 0)

    candidate_scores = np.where(
        valid_region,
        distance,
        -1,
    )

    y, x = np.unravel_index(
        np.argmax(candidate_scores),
        candidate_scores.shape,
    )

    return int(x), int(y)

**Build validation instance manifest**

In [17]:
instance_records = []

for label_path in sorted(VAL_LABELS_DIR.glob("*.txt")):
    image_path = validation_image_index.get(label_path.stem)

    if image_path is None:
        continue

    image, masks = load_instance_masks(
        image_path,
        label_path,
    )

    height, width = image.shape[:2]

    for instance_index, mask in enumerate(masks):
        foreground_pixels = int(mask.sum())

        instance_records.append(
            {
                "image_path": str(image_path),
                "label_path": str(label_path),
                "image_name": image_path.name,
                "instance_index": instance_index,
                "foreground_pixels": foreground_pixels,
                "foreground_ratio": foreground_pixels / (height * width),
            }
        )

In [18]:
instance_manifest = pd.DataFrame(instance_records)

print("Validation instances:", len(instance_manifest))
print(
    instance_manifest[
        [
            "foreground_pixels",
            "foreground_ratio",
        ]
    ].describe()
)

Validation instances: 249
       foreground_pixels  foreground_ratio
count         249.000000        249.000000
mean         2822.261044          0.016308
std          2054.272774          0.011871
min            21.000000          0.000121
25%          1310.000000          0.007570
50%          2562.000000          0.014804
75%          3779.000000          0.021837
max         11971.000000          0.069174


**Select nine representative instances**

In [19]:
lower_boundary = (
    instance_manifest["foreground_pixels"].quantile(0.33)
)

upper_boundary = (
    instance_manifest["foreground_pixels"].quantile(0.67)
)

In [20]:
def assign_size_group(pixel_count):
    if pixel_count <= lower_boundary:
        return "small"

    if pixel_count <= upper_boundary:
        return "medium"

    return "large"


instance_manifest["size_group"] = instance_manifest["foreground_pixels"].apply(
    assign_size_group
)

In [21]:
selected_groups = []

for size_group in ["small", "medium","large"]:
    group = (
        instance_manifest[instance_manifest["size_group"] == size_group]
        .sort_values("foreground_pixels")
        .reset_index(drop=True)
    )

    selected_indices = np.linspace(
        0, len(group) - 1,
        3, dtype=int
    )

    selected_groups.append(group.iloc[selected_indices])

In [22]:
selected_instances = pd.concat(selected_groups, ignore_index=True)

selection_path = OUTPUT_ROOT / "sam2_selected_instances.csv"
selected_instances.to_csv(selection_path, index=False)

selected_instances[
    [
        "image_name",
        "instance_index",
        "size_group",
        "foreground_pixels",
        "foreground_ratio",
    ]
]

,image_name,instance_index,size_group,foreground_pixels,foreground_ratio
0,2352.rf.13d894635b7d1c9765f6d1a2404156b7.jpg,1,small,21,0.000121
1,1971.rf.d029ad8edf2ff5e40a416a1399e143ae.jpg,3,small,894,0.005166
2,3228.rf.2bc533ca5e879aecb669ac56ea1c2373.jpg,0,small,1679,0.009702
3,2244.rf.e5270b51e6e7ea41debaf81f16273d7e.jpg,0,medium,1683,0.009725
4,1939.rf.3395d327f8a49d82491b18e04080fee7.jpg,1,medium,2562,0.014804
5,2040.rf.d10ea68f663f198606ec2a816fb7c35b.jpg,0,medium,3300,0.019069
6,3224.rf.ad00820ac18b68d77dc7e662e000df08.jpg,0,large,3308,0.019115
7,3066.rf.3e5fa3c6ece0d8f7337cdf619089fb27.jpg,0,large,4417,0.025524
8,3028.rf.323af1e3d52cd8dfcc7c01e46ce5947c.jpg,0,large,11971,0.069174
